# Business Entity Resolution — Amazon ML Challenge

Pipeline: **normalise → multi-channel TF-IDF blocking (S2/S3 → S1) → LightGBM pair matcher → assignment-aware F0.5 decoding**.

Outputs `output/matching_results.tsv` (leaderboard file) and `output/candidate_pairs.tsv` (exact set scored by the model). Runtime on a Kaggle CPU session: a few hours (blocking dominates). Intermediate results are cached in `work/`, so re-running a cell skips finished stages.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn anyascii lightgbm polars

## 1. Setup & configuration
Set `ER_DATA_DIR` if the dataset is not auto-detected under `/kaggle/input`.

In [ ]:
import glob, multiprocessing, os
from pathlib import Path


def _find_data_dir():
    if os.environ.get("ER_DATA_DIR"):
        return Path(os.environ["ER_DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
    return Path(hits[0]).parents[1] if hits else Path("dataset")


class config:
    DATA_DIR = _find_data_dir()                 # folder containing train/ and test/
    WORK_DIR = Path(os.environ.get("ER_WORK_DIR", "/kaggle/working/work" if os.path.isdir("/kaggle") else "work"))
    OUTPUT_DIR = Path(os.environ.get("ER_OUTPUT_DIR", "/kaggle/working/output" if os.path.isdir("/kaggle") else "output"))
    N_THREADS = os.cpu_count() or 4
    SEED = 42
    SAMPLE_FRAC = float(os.environ.get("ER_SAMPLE", 0))  # >0 only for quick smoke tests

    # blocking
    TOPK_FULL, TOPK_NAME, TOPK_CHAR = 8, 4, 4   # retrieved per S2/S3 record per channel
    TOPK_KEEP = (4, 2, 2)                       # kept after exact re-scoring (full/name/char rank)
    DF_CAP_FRAC, DF_CAP_MIN, DF_CAP_CHAR = 0.01, 1000, 0.002
    QUERY_CHUNK = 1_000_000

    # training
    TRAIN_QUERY_FRAC = 0.15   # train S2/S3 queries used to fit the model
    VAL_S1_FRAC = 0.05        # train S1 held out for validation


multiprocessing.set_start_method("fork", force=True)
config.WORK_DIR.mkdir(parents=True, exist_ok=True)
print("data:", config.DATA_DIR, "| cpus:", config.N_THREADS)


## 2. I/O

In [ ]:
"""TSV reading / submission writing."""
from pathlib import Path

import polars as pl



def read_tsv(path: Path) -> pl.DataFrame:
    return pl.read_csv(path, separator="\t", quote_char=None, infer_schema=False,
                       missing_utf8_is_empty_string=True)


def load_split(split: str) -> pl.DataFrame:
    """Concatenate the three sources of a split with a numeric `src` column."""
    d = config.DATA_DIR / split
    frames = []
    for s in (1, 2, 3):
        df = read_tsv(d / f"{split}_source{s}.tsv").with_columns(pl.lit(s, pl.Int8).alias("src"))
        frames.append(df)
    df = pl.concat(frames)
    if config.SAMPLE_FRAC:  # smoke-test mode: a slice of S1 + their matches + some distractors
        keep = (pl.col("entity_id").hash(1) % 10_000) < config.SAMPLE_FRAC * 10_000
        s1 = df.filter((pl.col("src") == 1) & keep)
        if split == "train":
            m = load_ground_truth().join(s1.select(pl.col("entity_id").alias("s1_id")), on="s1_id")["match_id"]
            keep = keep | pl.col("entity_id").is_in(m.implode())
        df = df.filter(keep)
    df = df.with_columns(
        [pl.col(c).fill_null("") for c in ("business_name", "business_address", "country")])
    return df.with_row_index("idx")


def load_ground_truth() -> pl.DataFrame:
    """Return long (s1_id, match_id) pairs."""
    gt = read_tsv(config.DATA_DIR / "train" / "train_ground_truth.tsv")
    return (gt.rename({"source1_entity_id": "s1_id"})
              .with_columns(pl.col("matched_entity_ids").fill_null("").str.split(","))
              .explode("matched_entity_ids")
              .rename({"matched_entity_ids": "match_id"})
              .filter(pl.col("match_id").str.len_chars() > 0))


def write_lists(path: Path, s1: pl.DataFrame, pairs: pl.DataFrame, col: str):
    """s1: (idx, entity_id) of every Source-1 record in output order.
    pairs: (s, q_id) long frame. Writes one row per S1 with a comma-joined, de-duplicated list."""
    path.parent.mkdir(parents=True, exist_ok=True)
    lists = pairs.unique(["s", "q_id"]).group_by("s").agg(pl.col("q_id").str.join(",").alias(col))
    out = (s1.join(lists, left_on="idx", right_on="s", how="left", maintain_order="left")
             .select(pl.col("entity_id").alias("source1_entity_id"), pl.col(col).fill_null("")))
    assert out.height == s1.height
    out.write_csv(path, separator="\t", quote_style="never")


## 3. Normalisation
Inverts the observed noise generators (accents, native scripts, abbreviations, aliases, junk, state variants, zero-padded numbers).

In [ ]:
"""Name / address canonicalisation.

Every rule inverts one of the observed noise generators: accents, native
scripts, casing, punctuation/junk prefixes, honorifics, legal-suffix and
street-type abbreviations, domain names, DBA/"t/a" aliases, state
abbreviations and zero-padded house numbers.  All heavy lifting is done with
vectorised polars / pyarrow string kernels; only non-Latin strings go through
Python (anyascii romanisation), and only once per unique value.
"""
from multiprocessing import Pool

import polars as pl
import pyarrow.compute as pc
from anyascii import anyascii


# --------------------------------------------------------------------------
# token maps (applied on whole tokens)
# --------------------------------------------------------------------------
NAME_MAP = {
    "pvt": "private", "pvtltd": "private limited", "ltd": "limited", "corp": "corporation",
    "co": "company", "inc": "incorporated", "incorp": "incorporated", "intl": "international",
    "mfg": "manufacturing", "assoc": "association", "assn": "association", "bros": "brothers",
    "svcs": "services", "svc": "services", "mgmt": "management", "cie": "compagnie",
    "ste": "societe", "et": "and", "centre": "center", "l l c": "llc", "l l p": "llp",
    "limited liability company": "llc", "etablissements": "ets", "ets": "ets",
    # honorifics / fillers -> removed
    "the": "", "mr": "", "mrs": "", "ms": "", "dr": "", "sri": "", "shri": "", "smt": "",
    "messrs": "", "na": "",
}
LEGAL = {"private", "limited", "corporation", "company", "incorporated", "llc", "llp", "lp",
         "plc", "pllc", "sarl", "sas", "sasu", "eurl", "sa", "sci", "gmbh", "ltda", "pc", "ets"}
ALIAS_MARKERS = r"\b(?:dba|formerly|aka|trading as|t a|d b a|a k a|fka)\b"

ADDR_MAP = {
    "street": "st", "saint": "st", "road": "rd", "drive": "dr", "drvie": "dr", "avenue": "ave",
    "av": "ave", "lane": "ln", "boulevard": "blvd", "bd": "blvd", "bvd": "blvd", "court": "ct",
    "circle": "cir", "place": "pl", "highway": "hwy", "parkway": "pkwy", "pky": "pkwy",
    "square": "sq", "terrace": "ter", "trail": "trl", "mount": "mt", "fort": "ft", "north": "n",
    "south": "s", "east": "e", "west": "w", "suite": "ste", "apartment": "apt", "floor": "fl",
    "rue": "r", "chemin": "ch", "impasse": "imp", "allee": "all", "route": "rte", "sector": "sec",
    "near": "nr", "opposite": "opp", "opp": "opp", "centre": "center", "ctr": "center",
    "expressway": "expy", "freeway": "fwy", "point": "pt", "heights": "hts", "junction": "jct",
    "cross": "crs", "main": "main", "building": "bldg", "bldg": "bldg",
    # house-number prefixes / fillers -> removed
    "no": "", "hno": "", "h no": "", "plot": "", "khno": "", "kh no": "", "kh": "", "na": "",
    "door": "", "flat": "", "shop": "", "num": "", "number": "",
}
US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia",
    "kansas": "ks", "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms",
    "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv", "new hampshire": "nh",
    "new jersey": "nj", "new mexico": "nm", "new york": "ny", "north carolina": "nc",
    "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or", "pennsylvania": "pa",
    "rhode island": "ri", "south carolina": "sc", "south dakota": "sd", "tennessee": "tn",
    "texas": "tx", "utah": "ut", "vermont": "vt", "virginia": "va", "washington": "wa",
    "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy", "district of columbia": "dc",
    "puerto rico": "pr",
}
IN_STATES = {
    "up": "uttar pradesh", "wb": "west bengal", "mh": "maharashtra", "ka": "karnataka",
    "tn": "tamil nadu", "dl": "delhi", "gj": "gujarat", "rj": "rajasthan", "kl": "kerala",
    "ap": "andhra pradesh", "ts": "telangana", "tg": "telangana", "hr": "haryana",
    "pb": "punjab", "mp": "madhya pradesh", "br": "bihar", "od": "odisha", "or": "odisha",
    "orissa": "odisha", "jh": "jharkhand", "cg": "chhattisgarh", "ct": "chhattisgarh",
    "uk": "uttarakhand", "ut": "uttarakhand", "hp": "himachal pradesh", "as": "assam",
    "ga": "goa", "jk": "jammu and kashmir", "ch": "chandigarh", "py": "puducherry",
    "pondicherry": "puducherry", "nct of delhi": "delhi", "new delhi": "delhi",
    "bangalore": "bengaluru", "bombay": "mumbai", "madras": "chennai", "calcutta": "kolkata",
    "gurgaon": "gurugram",
}


def map_tokens(e: pl.Expr, mapping: dict) -> pl.Expr:
    """Whole-token replacement using Aho-Corasick (tokens separated by 2 spaces)."""
    keys = [" " + k.replace(" ", "  ") + " " for k in mapping]
    vals = [" " + v.replace(" ", "  ") + " " if v else " " for v in mapping.values()]
    e = pl.lit(" ") + e.str.replace_all(" ", "  ", literal=True) + pl.lit(" ")
    e = e.str.replace_many(keys, vals)
    return e.str.replace_all(r"\s+", " ").str.strip_chars()


def _romanise(values):
    return [anyascii(v) for v in values]


def fold(s: pl.Series) -> pl.Series:
    """Romanise non-Latin scripts, strip accents, lower-case."""
    s = s.fill_null("")
    uniq = s.filter(s.str.contains(r"[^\p{Latin}\p{Common}\p{Inherited}]")).unique().to_list()
    if uniq:
        chunks = [uniq[i::config.N_THREADS] for i in range(config.N_THREADS)]
        with Pool(config.N_THREADS) as p:
            outs = p.map(_romanise, chunks)
        lut = pl.DataFrame({"k": [x for c in chunks for x in c], "v": [x for o in outs for x in o]})
        s = (pl.DataFrame({"k": s}).join(lut, on="k", how="left", maintain_order="left")
             .select(pl.coalesce("v", "k")).to_series())
    arr = pc.utf8_normalize(s.to_arrow(), form="NFKD")
    return pl.Series(arr).str.replace_all(r"\p{M}", "").str.to_lowercase()


def _basic_clean(e: pl.Expr) -> pl.Expr:
    return (e.str.replace_all("&", " and ", literal=True)
             .str.replace_all(r"[.'`’]", "")
             .str.replace_all(r"[^a-z0-9]+", " ")
             .str.strip_chars())


def normalize(df: pl.DataFrame) -> pl.DataFrame:
    """Add normalised columns to a frame with business_name/business_address/country."""
    df = df.with_columns(fold(df["business_name"]).alias("_n"), fold(df["business_address"]).alias("_a"))

    # ---------------- names ----------------
    n = (pl.col("_n").str.replace_all(r"^\s*www\.", "")
         .str.replace_all(r"\.(?:co\.in|com|net|org|in|fr|biz|info|io|co|us)\b", " ")
         .str.replace_all(r"\bm/s\b", " ")
         .str.replace_all(r"\b(?:t/a|d/b/a|a/k/a|f/k/a)\b", " dba "))
    df = df.with_columns(map_tokens(_basic_clean(n), NAME_MAP).alias("name"))

    # ---------------- addresses ----------------
    a = _basic_clean(pl.col("_a").str.replace_all(r"\bn/a\b", " "))
    a = pl.when(pl.col("country") == "US").then(map_tokens(a, US_STATES)) \
          .when(pl.col("country") == "India").then(map_tokens(a, IN_STATES)).otherwise(a)
    a = map_tokens(a, ADDR_MAP)
    a = a.str.replace_all(r"\b0+(\d)", "$1")
    df = df.with_columns(a.alias("addr"), nonlatin(pl.col("business_name")).alias("nl_name"),
                         nonlatin(pl.col("business_address")).alias("nl_addr"))
    return df.drop("_n", "_a")


def nonlatin(e: pl.Expr) -> pl.Expr:
    return e.str.contains(r"[^\p{Latin}\p{Common}\p{Inherited}]")


def derive(df: pl.DataFrame) -> pl.DataFrame:
    """Alias split, legal-suffix split, space-less core and house numbers."""
    df = df.with_columns(pl.col("name").str.replace(ALIAS_MARKERS, "\x00").alias("_al"))
    parts = pl.col("_al").str.split_exact("\x00", 1)
    df = df.with_columns(
        parts.struct.field("field_0").str.strip_chars().alias("alias_a"),
        parts.struct.field("field_1").fill_null("").str.replace_all(ALIAS_MARKERS, " ")
        .str.replace_all(r"\s+", " ").str.strip_chars().alias("alias_b"),
        pl.col("name").str.replace_all(ALIAS_MARKERS, " ").str.replace_all(r"\s+", " ")
        .str.strip_chars().alias("name"),
    )
    legal_re = r"\b(?:" + "|".join(sorted(LEGAL, key=len, reverse=True)) + r")\b"
    df = df.with_columns(
        pl.col("name").str.replace_all(legal_re, " ").str.replace_all(r"\s+", " ")
        .str.strip_chars().alias("core"),
        pl.col("name").str.extract_all(legal_re).list.unique().list.sort().list.join(" ").alias("legal"),
    )
    df = df.with_columns(pl.when(pl.col("core") == "").then(pl.col("name")).otherwise(pl.col("core")).alias("core"))
    df = df.with_columns(pl.col("core").str.replace_all(" ", "", literal=True).alias("nsp"),
                         pl.col("addr").str.extract_all(r"\b\d+\b").alias("nums"))
    return df.drop("_al")


## 4. Lexicons mined from training pairs

In [ ]:
"""Dictionaries mined from the TRAINING pairs only (no external data).

* name lexicon   : romanised native-script token -> Latin token. Aligned by
                   position on matched pairs with equal token counts
                   (e.g. "praivet" -> "private", "phorcyun" -> "fortune").
* address lexicon: romanised native-script address token -> S1 address tokens
                   that co-occur with it in >= 85% of matched pairs
                   (e.g. native "Maharashtra" -> "maharashtra").
* segmentation   : glued tokens / domain names ("elberteducation") are split
                   into S1 name-vocabulary words ("elbert education").
"""
import math
from multiprocessing import Pool

import polars as pl



def _tok(col):
    return pl.col(col).str.split(" ").list.eval(pl.element().filter(pl.element() != ""))


def mine(norm: pl.DataFrame, gt: pl.DataFrame) -> dict:
    ids = norm.select("entity_id", "name", "addr", "nl_name", "nl_addr", "business_address")
    pairs = (gt.join(ids.rename(lambda c: "s_" + c), left_on="s1_id", right_on="s_entity_id")
               .join(ids.rename(lambda c: "q_" + c), left_on="match_id", right_on="q_entity_id"))
    # ---- names: positional alignment ----
    p = (pairs.filter(pl.col("q_nl_name"))
         .select(_tok("q_name").alias("qt"), _tok("s_name").alias("st"))
         .filter(pl.col("qt").list.len() == pl.col("st").list.len())
         .explode(["qt", "st"]))
    cnt = p.group_by("qt", "st").len("n")
    tot = cnt.group_by("qt").agg(pl.col("n").sum().alias("tot"))
    name_lex = (cnt.sort("n", descending=True).group_by("qt").first().join(tot, on="qt")
                .filter((pl.col("n") >= 2) & (pl.col("n") >= 0.5 * pl.col("tot")) & (pl.col("qt") != pl.col("st")))
                .select(pl.col("qt").alias("k"), pl.col("st").alias("v")))
    # ---- addresses: co-occurrence of romanised native segments ----
    a = pairs.filter(pl.col("q_nl_addr")).select(
        pl.col("q_business_address").str.extract_all(r"[^\p{Latin}\p{Common}\p{Inherited}][^,]*").list.join(" ").alias("seg"),
        _tok("s_addr").list.unique().alias("st"))
    a = a.with_columns(fold(a["seg"]).alias("seg")).with_columns(
        _basic_clean(pl.col("seg")).str.split(" ").list.unique().alias("qt"))
    a = a.with_row_index("r")
    qx = a.select("r", "qt").explode("qt").filter(pl.col("qt") != "")
    tot = qx.group_by("qt").len("tot")
    co = (qx.join(a.select("r", "st").explode("st"), on="r").group_by("qt", "st").len("n")
          .join(tot, on="qt").filter((pl.col("tot") >= 5) & (pl.col("n") >= 0.85 * pl.col("tot"))))
    addr_lex = (co.sort("n", descending=True).group_by("qt").agg(pl.col("st").head(3).str.join(" ").alias("v"))
                .rename({"qt": "k"}).filter(pl.col("k") != pl.col("v")))
    print(f"[lexicon] name entries {name_lex.height:,}  address entries {addr_lex.height:,}", flush=True)
    return {"name": name_lex, "addr": addr_lex}


def map_col(df: pl.DataFrame, col: str, lut: pl.DataFrame, mask: pl.Expr) -> pl.DataFrame:
    """Token-wise replacement of `col` for rows where `mask` holds."""
    if lut.height == 0:
        return df
    sub = df.filter(mask.fill_null(False)).select("idx", col)
    if sub.height == 0:
        return df
    ex = (sub.with_columns(pl.col(col).str.split(" ")).explode(col).with_row_index("_o")
          .join(lut, left_on=col, right_on="k", how="left")
          .with_columns(pl.coalesce("v", col).alias(col)).sort("_o")
          .group_by("idx", maintain_order=True).agg(pl.col(col).str.join(" ").str.strip_chars().alias("_new")))
    return (df.join(ex, on="idx", how="left")
              .with_columns(pl.coalesce("_new", col).alias(col)).drop("_new"))


# ---------------------------------------------------------------- segmentation
_VOCAB: dict = {}


def _init(v):
    global _VOCAB
    _VOCAB = v


def _segment(words):
    out = []
    for w in words:
        n = len(w)
        best = [(-1e9, None)] * (n + 1)
        best[0] = (0.0, [])
        for i in range(n):
            if best[i][1] is None:
                continue
            for j in range(i + 2, min(n, i + 20) + 1):
                f = _VOCAB.get(w[i:j])
                if f:
                    sc = best[i][0] + math.log(f) - 8.0
                    if sc > best[j][0]:
                        best[j] = (sc, best[i][1] + [w[i:j]])
        seg = best[n][1]
        out.append(" ".join(seg) if seg and 2 <= len(seg) <= 5 else None)
    return out


def segment_lut(norm: pl.DataFrame) -> pl.DataFrame:
    s1_tok = norm.filter(pl.col("src") == 1).select(_tok("name").alias("t")).explode("t")
    vc = s1_tok.group_by("t").len("n").filter((pl.col("n") >= 3) & (pl.col("t").str.len_chars() >= 2))
    vocab = dict(zip(vc["t"].to_list(), vc["n"].to_list()))
    cand = (norm.filter(pl.col("src") != 1).select(_tok("name").alias("t")).explode("t").unique()
            .filter((pl.col("t").str.len_chars() >= 8) & ~pl.col("t").str.contains(r"\d"))
            .filter(~pl.col("t").is_in(vc["t"].implode())))["t"].to_list()
    chunks = [cand[i::config.N_THREADS] for i in range(config.N_THREADS)]
    with Pool(config.N_THREADS, initializer=_init, initargs=(vocab,)) as p:
        outs = p.map(_segment, chunks)
    lut = pl.DataFrame({"k": [x for c in chunks for x in c], "v": [x for o in outs for x in o]},
                       schema={"k": pl.String, "v": pl.String}).drop_nulls()
    print(f"[lexicon] segmented {lut.height:,} of {len(cand):,} glued tokens", flush=True)
    return lut


def apply(norm: pl.DataFrame, lex: dict) -> pl.DataFrame:
    norm = map_col(norm, "name", lex["name"], pl.col("nl_name"))
    norm = map_col(norm, "addr", lex["addr"], pl.col("nl_addr"))
    norm = map_col(norm, "name", segment_lut(norm), pl.col("src") != 1)
    return norm


## 5. Pre-processing

In [ ]:
"""Stage 1: load a split, normalise, apply train-mined lexicons, cache as parquet."""
import sys
import time

import polars as pl


COLS = ["idx", "entity_id", "src", "country", "name", "core", "alias_a", "alias_b", "legal",
        "nsp", "addr", "nums", "nl_name", "nl_addr", "business_name", "business_address"]


def _base(split: str) -> pl.DataFrame:
    out = config.WORK_DIR / f"{split}_base.parquet"
    if out.exists():
        return pl.read_parquet(out)
    t = time.time()
    df = load_split(split)
    df = pl.concat([normalize(df.slice(i, 2_000_000)) for i in range(0, df.height, 2_000_000)])
    config.WORK_DIR.mkdir(parents=True, exist_ok=True)
    df.write_parquet(out)
    print(f"[prep] {split}: normalised {df.height:,} records in {time.time() - t:.0f}s", flush=True)
    return df


def _lexicon() -> dict:
    paths = {k: config.WORK_DIR / f"lexicon_{k}.parquet" for k in ("name", "addr")}
    if all(p.exists() for p in paths.values()):
        return {k: pl.read_parquet(p) for k, p in paths.items()}
    lex = mine(_base("train"), load_ground_truth())
    for k, p in paths.items():
        lex[k].write_parquet(p)
    return lex


def prep(split: str) -> pl.DataFrame:
    out = config.WORK_DIR / f"{split}_norm.parquet"
    if out.exists():
        return pl.read_parquet(out)
    lex = _lexicon()
    t = time.time()
    df = derive(apply(_base(split), lex)).select(COLS)
    df.write_parquet(out)
    print(f"[prep] {split}: lexicon + derive in {time.time() - t:.0f}s", flush=True)
    return df


In [ ]:
for split in ('train', 'test'):
    prep(split)

## 6. Blocking / candidate generation

In [ ]:
"""Stage 2: candidate generation.

Blocking is done from the S2/S3 side (every S2/S3 record belongs to at most
one S1), per country (country is treated as an open set of labels).  Three
sparse TF-IDF retrieval channels are unioned:

* full : word tokens of core name (tagged n:) + address (tagged a:)
* name : word tokens of the core name only
* char : character 3-grams of the space-less core name (typos, domains, glued words)

Tokens whose S1 document frequency is above a cap are left out of the
retrieval matrix (they are uninformative and dominate the cost) but still
count in the vector norms.  For each surviving (query, S1) pair the exact
cosine of every channel (+ an address-only channel) is computed and stored as
features.
"""
import time

import numpy as np
import polars as pl
import scipy.sparse as sp
from sparse_dot_topn import sp_matmul_topn


CHANNELS = ("full", "name", "char", "addr")
RETRIEVE = {"full": config.TOPK_FULL, "name": config.TOPK_NAME, "char": config.TOPK_CHAR}


def run(split: str) -> pl.DataFrame:
    out = config.WORK_DIR / f"cand_{split}.parquet"
    if out.exists():
        return pl.read_parquet(out)
    norm = pl.read_parquet(config.WORK_DIR / f"{split}_norm.parquet")
    cand = block(norm)
    cand.write_parquet(out)
    return cand


def _words(col: str, tag: str) -> pl.Expr:
    return (pl.col(col).str.split(" ").list.eval(pl.element().filter(pl.element().str.len_chars() > 0))
            .list.eval(pl.lit(tag) + pl.element()))


def _trigrams(col: str, maxlen: int = 40) -> pl.Expr:
    s = pl.lit("#") + pl.col(col) + pl.lit("#")
    grams = pl.concat_list([s.str.slice(i, 3) for i in range(maxlen)])
    return grams.list.eval(pl.element().filter(pl.element().str.len_chars() == 3))


def token_lists(df: pl.DataFrame) -> pl.DataFrame:
    return df.select(
        pl.concat_list(_words("core", "n"), _words("addr", "a")).alias("full"),
        _words("core", "n").alias("name"),
        _trigrams("nsp").alias("char"),
        _words("addr", "a").alias("addr"),
    )


def _vocab(tok: pl.Series, n_docs: int, cap_frac: float) -> pl.DataFrame:
    v = (pl.DataFrame({"r": np.arange(len(tok), dtype=np.uint32), "tok": tok})
         .explode("tok").drop_nulls().unique().group_by("tok").len("df"))
    cap = max(config.DF_CAP_MIN, int(cap_frac * n_docs))
    return v.with_columns(
        (np.log(n_docs / pl.col("df")) + 1.0).cast(pl.Float32).alias("idf"),
        (pl.col("df") <= cap).alias("keep"),
    ).with_row_index("tid")


def _matrix(tok: pl.Series, vocab: pl.DataFrame):
    """Return (full L2-normalised csr, retrieval csr restricted to kept tokens)."""
    n = len(tok)
    ex = (pl.DataFrame({"r": np.arange(n, dtype=np.uint32), "tok": tok})
          .explode("tok").drop_nulls().unique()
          .join(vocab, on="tok", how="inner"))
    norm = ex.group_by("r").agg((pl.col("idf") ** 2).sum().sqrt().alias("nrm"))
    ex = ex.join(norm, on="r").with_columns((pl.col("idf") / pl.col("nrm")).alias("w"))
    r, c, w, k = (ex[x].to_numpy() for x in ("r", "tid", "w", "keep"))
    V = vocab.height
    full = sp.csr_matrix((w.astype(np.float32), (r, c)), shape=(n, V))
    ret = sp.csr_matrix((w[k].astype(np.float32), (r[k], c[k])), shape=(n, V))
    return full, ret


def pair_cos(A: sp.csr_matrix, B: sp.csr_matrix, ia: np.ndarray, ib: np.ndarray, chunk=2_000_000):
    out = np.empty(len(ia), np.float32)
    for i in range(0, len(ia), chunk):
        out[i:i + chunk] = np.asarray(A[ia[i:i + chunk]].multiply(B[ib[i:i + chunk]]).sum(1)).ravel()
    return out


def block_country(s1: pl.DataFrame, q: pl.DataFrame) -> pl.DataFrame:
    """s1/q: normalised frames of one country. Returns pruned candidate pairs with cosines."""
    t1 = token_lists(s1)
    s1_idx = s1["idx"].to_numpy()
    vocab, f1, bt = {}, {}, {}
    for ch in CHANNELS:
        vocab[ch] = _vocab(t1[ch], s1.height, config.DF_CAP_CHAR if ch == "char" else config.DF_CAP_FRAC)
        f1[ch], r1 = _matrix(t1[ch], vocab[ch])
        if ch in RETRIEVE:
            bt[ch] = r1.T.tocsr()
    del t1
    out = []
    for i in range(0, q.height, config.QUERY_CHUNK):
        qc = q.slice(i, config.QUERY_CHUNK)
        tq = token_lists(qc)
        fq, res = {}, []
        for ch in CHANNELS:
            fq[ch], rq = _matrix(tq[ch], vocab[ch])
            if ch in RETRIEVE:
                m = sp_matmul_topn(rq, bt[ch], top_n=RETRIEVE[ch], threshold=0.05,
                                   n_threads=config.N_THREADS).tocoo()
                res.append(pl.DataFrame({"qi": m.row.astype(np.uint32), "si": m.col.astype(np.uint32)}))
        cand = pl.concat(res).unique()
        qi, si = cand["qi"].to_numpy(), cand["si"].to_numpy()
        cand = cand.with_columns(*[pl.Series(f"cos_{ch}", pair_cos(fq[ch], f1[ch], qi, si)) for ch in CHANNELS])
        rk = {ch: pl.col(f"cos_{ch}").rank("ordinal", descending=True).over("qi") for ch in RETRIEVE}
        keep = pl.lit(False)
        for ch, k in zip(RETRIEVE, config.TOPK_KEEP):
            keep = keep | (rk[ch] <= k)
        cand = cand.filter(keep)
        qmap = qc["idx"].to_numpy()
        out.append(cand.select(pl.Series("q", qmap[cand["qi"].to_numpy()]),
                               pl.Series("s", s1_idx[cand["si"].to_numpy()]),
                               *[f"cos_{ch}" for ch in CHANNELS]))
    return pl.concat(out)


def block(norm: pl.DataFrame, query_mask: pl.Expr | None = None) -> pl.DataFrame:
    s1_all = norm.filter(pl.col("src") == 1)
    q_all = norm.filter(pl.col("src") != 1)
    if query_mask is not None:
        q_all = q_all.filter(query_mask)
    out = []
    for c in s1_all["country"].unique().sort().to_list():
        t = time.time()
        s1 = s1_all.filter(pl.col("country") == c)
        q = q_all.filter(pl.col("country") == c)
        if q.height == 0 or s1.height == 0:
            continue
        cand = block_country(s1, q)
        out.append(cand)
        print(f"[block] {c}: {s1.height:,} S1 x {q.height:,} queries -> {cand.height:,} pairs "
              f"({cand.height / q.height:.1f}/query) {time.time() - t:.0f}s", flush=True)
    # queries whose country label has no S1 at all fall back to the global S1 pool
    orphan = q_all.filter(~pl.col("country").is_in(s1_all["country"].unique().implode()))
    if orphan.height:
        out.append(block_country(s1_all, orphan))
    return pl.concat(out)


In [ ]:
for split in ('train', 'test'):
    run(split)

## 7. Features

In [ ]:
"""Stage 3: pairwise + context features for candidate pairs.

Country is deliberately NOT a feature so that the model transfers to unseen
countries (France in test).
"""
import time

import numpy as np
import polars as pl
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist


CHUNK = 2_000_000


def _cp(a, b, scorer):
    return cpdist(a, b, scorer=scorer, workers=config.N_THREADS, dtype=np.float32)


def table(norm: pl.DataFrame) -> pl.DataFrame:
    """Per-record columns needed by the features, row position == idx."""
    norm = norm.sort("idx")
    assert norm["idx"][-1] == norm.height - 1
    s1 = norm.filter(pl.col("src") == 1)
    core_cnt = s1.group_by("country", "core").len("core_cnt")
    t = norm.select("idx", "src", "country", "name", "core", "nsp", "alias_a", "alias_b", "legal", "addr", "nums")
    t = t.join(core_cnt, on=["country", "core"], how="left", maintain_order="left").with_columns(
        pl.col("core_cnt").fill_null(0).cast(pl.Float32))
    return t.sort("idx")


def _pair_chunk(c: pl.DataFrame, T: pl.DataFrame) -> pl.DataFrame:
    qi, si = c["q"].to_numpy(), c["s"].to_numpy()
    Q, S = T[qi], T[si]
    f = {}
    qn, sn = Q["name"].to_list(), S["name"].to_list()
    qc, sc = Q["core"].to_list(), S["core"].to_list()
    f["n_ratio"] = _cp(qn, sn, fuzz.ratio)
    f["n_tsort"] = _cp(qn, sn, fuzz.token_sort_ratio)
    f["n_tset"] = _cp(qc, sc, fuzz.token_set_ratio)
    f["n_partial"] = _cp(qc, sc, fuzz.partial_ratio)
    f["n_jw"] = _cp(qc, sc, JaroWinkler.normalized_similarity)
    qs_, ss_ = Q["nsp"].to_list(), S["nsp"].to_list()
    f["nsp_ratio"] = _cp(qs_, ss_, fuzz.ratio)
    f["nsp_partial"] = _cp(qs_, ss_, fuzz.partial_ratio)
    has_alias = (Q["alias_b"].str.len_chars() > 0).to_numpy()
    al_b = _cp(Q["alias_b"].to_list(), sc, fuzz.token_sort_ratio)
    al_a = _cp(Q["alias_a"].to_list(), sc, fuzz.token_sort_ratio)
    f["alias_best"] = np.where(has_alias, np.maximum(al_a, al_b), -1).astype(np.float32)
    qa, sa = Q["addr"].to_list(), S["addr"].to_list()
    f["a_ratio"] = _cp(qa, sa, fuzz.ratio)
    f["a_tset"] = _cp(qa, sa, fuzz.token_set_ratio)
    f["a_partial"] = _cp(qa, sa, fuzz.partial_ratio)
    out = pl.DataFrame(f).with_columns(
        pl.Series("q_core_len", Q["core"].str.len_chars()), pl.Series("s_core_len", S["core"].str.len_chars()),
        pl.Series("q_ntok", Q["core"].str.count_matches(" ") + 1), pl.Series("s_ntok", S["core"].str.count_matches(" ") + 1),
        pl.Series("q_addr_len", Q["addr"].str.len_chars()), pl.Series("s_addr_len", S["addr"].str.len_chars()),
        pl.Series("first_tok_eq", (Q["core"].str.split(" ").list.first() == S["core"].str.split(" ").list.first()).cast(pl.Int8)),
        pl.Series("legal_eq", (Q["legal"] == S["legal"]).cast(pl.Int8)),
        pl.Series("legal_conf", ((Q["legal"] != S["legal"]) & (Q["legal"] != "") & (S["legal"] != "")).cast(pl.Int8)),
        pl.Series("q_legal_empty", (Q["legal"] == "").cast(pl.Int8)),
        pl.Series("s_core_cnt", S["core_cnt"]), pl.Series("q_core_cnt", Q["core_cnt"]),
        pl.Series("src", Q["src"]),
        pl.Series("q_nnum", Q["nums"].list.len()), pl.Series("s_nnum", S["nums"].list.len()),
        pl.Series("firstnum_eq", (Q["nums"].list.first() == S["nums"].list.first()).fill_null(False).cast(pl.Int8)),
    )
    # house-number overlap via exploded join
    pid = np.arange(len(qi), dtype=np.uint32)
    qx = pl.DataFrame({"p": pid, "n": Q["nums"]}).explode("n").drop_nulls().unique()
    sx = pl.DataFrame({"p": pid, "n": S["nums"]}).explode("n").drop_nulls().unique()
    com = qx.join(sx, on=["p", "n"]).group_by("p").len("num_common")
    num_common = np.zeros(len(qi), np.float32)
    num_common[com["p"].to_numpy()] = com["num_common"].to_numpy()
    out = out.with_columns(pl.Series("num_common", num_common)).with_columns(
        ((pl.col("q_nnum") > 0) & (pl.col("s_nnum") > 0) & (pl.col("num_common") == 0)).cast(pl.Int8).alias("num_conflict"))
    return pl.concat([c, out], how="horizontal")


def _margins(cols, by):
    ex = []
    for col in cols:
        mx = pl.col(col).max().over(by)
        second = pl.col(col).top_k(2).min().over(by)
        ex += [(pl.col(col) - mx).alias(f"gap{by}_{col}"),
               (pl.col(col) - pl.when(pl.col(col) == mx).then(second).otherwise(mx)).alias(f"mar{by}_{col}")]
    return ex


def context_cos(cand: pl.DataFrame) -> pl.DataFrame:
    """Relative features from blocking cosines. Must run on the FULL candidate set of a split."""
    df = cand.with_columns((pl.col("cos_full") + 0.5 * pl.col("cos_name") + 0.5 * pl.col("cos_addr")).alias("comb"))
    df = df.with_columns(
        *_margins(("cos_full", "cos_name", "cos_char", "cos_addr", "comb"), "q"),
        *_margins(("comb",), "s"),
        pl.len().over("q").cast(pl.Int16).alias("q_ncand"),
        pl.len().over("s").cast(pl.Int16).alias("s_ncand"),
        pl.col("comb").rank("ordinal", descending=True).over("q").cast(pl.Int16).alias("rankq_comb"),
        pl.col("comb").rank("ordinal", descending=True).over("s").cast(pl.Int16).alias("ranks_comb"),
    )
    df = df.with_columns((pl.col("rankq_comb") == 1).cast(pl.Int8).alias("is_top"))
    return df.with_columns(pl.col("is_top").sum().over("s").cast(pl.Int16).alias("s_ntop"))


def build(cand: pl.DataFrame, T: pl.DataFrame, verbose=True) -> pl.DataFrame:
    """cand must already carry context_cos columns and contain every candidate of its queries.
    T = table(norm)."""
    t = time.time()
    parts = [_pair_chunk(cand.slice(i, CHUNK), T) for i in range(0, cand.height, CHUNK)]
    df = pl.concat(parts)
    df = df.with_columns(*_margins(("n_tset", "a_tset", "nsp_ratio"), "q"))
    if verbose:
        print(f"[features] {df.height:,} pairs, {len(feature_cols(df))} features in {time.time() - t:.0f}s", flush=True)
    return df


def feature_cols(df: pl.DataFrame) -> list:
    return [c for c in df.columns if c not in ("q", "s", "label", "country", "isval")]


## 8. Metric (macro F0.5)

In [ ]:
"""Macro F0.5 exactly as defined by the challenge (singletons included)."""
import polars as pl

BETA2 = 0.25


def f05(n_pred: int, n_true: int, tp: int) -> float:
    if n_true == 0:
        return 1.0 if n_pred == 0 else 0.0
    if n_pred == 0 or tp == 0:
        return 0.0
    p, r = tp / n_pred, tp / n_true
    return (1 + BETA2) * p * r / (BETA2 * p + r)


def macro_f05(pred: pl.DataFrame, truth: pl.DataFrame, s1_ids: pl.Series) -> dict:
    """pred/truth: long frames with columns (s, q). s1_ids: the evaluated S1 set."""
    s1 = pl.DataFrame({"s": s1_ids}).unique()
    pred = pred.select("s", "q").unique().join(s1, on="s", how="semi")
    truth = truth.select("s", "q").unique().join(s1, on="s", how="semi")
    tp = pred.join(truth, on=["s", "q"]).group_by("s").len("tp")
    df = (s1.join(pred.group_by("s").len("np"), on="s", how="left")
            .join(truth.group_by("s").len("nt"), on="s", how="left")
            .join(tp, on="s", how="left").fill_null(0))
    p = pl.col("tp") / pl.col("np")
    r = pl.col("tp") / pl.col("nt")
    f = (pl.when(pl.col("nt") == 0).then((pl.col("np") == 0).cast(pl.Float64))
           .when((pl.col("np") == 0) | (pl.col("tp") == 0)).then(0.0)
           .otherwise((1 + BETA2) * p * r / (BETA2 * p + r)))
    df = df.with_columns(f.alias("f"))
    single = df.filter(pl.col("nt") == 0)
    multi = df.filter(pl.col("nt") > 0)
    tot_tp, tot_p, tot_t = df["tp"].sum(), df["np"].sum(), df["nt"].sum()
    return {"macro_f05": df["f"].mean(), "n": df.height,
            "singleton_f": single["f"].mean() if single.height else None,
            "nonsingleton_f": multi["f"].mean() if multi.height else None,
            "micro_precision": tot_tp / max(tot_p, 1), "micro_recall": tot_tp / max(tot_t, 1)}


In [ ]:
assert abs(f05(3, 2, 2) - 0.714) < 1e-3 and f05(0, 0, 0) == 1.0 and f05(1, 0, 0) == 0.0
print('metric self-test OK')

## 9. Decoding

In [ ]:
"""Stage 5: turn pair probabilities into per-S1 match lists.

1. Assignment constraint - every S2/S3 record matches at most one S1, so a
   record may only be linked to its highest-probability S1 (and only if
   p >= tau).
2. Per-S1 expected-F0.5 prefix selection (optional): among the records assigned to an
   S1 (sorted by p), choose the prefix - possibly empty - that maximises the
   expected F0.5, which naturally leaves weak singletons empty.
"""
import numpy as np
import polars as pl


def assign(pp: pl.DataFrame, tau: float) -> pl.DataFrame:
    """pp: (q, s, p). Returns the (q, s, p) rows that survive the assignment constraint."""
    best = pp.sort(["q", "p"], descending=[False, True]).group_by("q", maintain_order=True).first()
    return best.filter(pl.col("p") >= tau)


def expected_f_prefix(pp: pl.DataFrame, tau: float, miss: float) -> pl.DataFrame:
    """Assignment, then keep the prefix of each S1's list that maximises E[F0.5].

    E[F] for prefix k>=1 ~= 1.25 * sum_{i<=k} p_i / (0.25 * (sum_all p + miss) + k)
    E[F] for k=0        ~= prod_i (1 - p_i) * exp(-miss)
    `miss` = expected true matches outside the assigned list (tuned on validation).
    """
    a = assign(pp, tau).sort(["s", "p"], descending=[False, True])
    a = a.with_columns(
        pl.col("p").cum_sum().over("s").alias("cp"),
        pl.int_range(1, pl.len() + 1).over("s").alias("k"),
        pl.col("p").sum().over("s").alias("tot"),
        (1 - pl.col("p")).log().sum().over("s").alias("log_none"),
    )
    a = a.with_columns((1.25 * pl.col("cp") / (0.25 * (pl.col("tot") + miss) + pl.col("k"))).alias("ef"))
    a = a.with_columns(pl.col("ef").max().over("s").alias("ef_best"),
                       (pl.col("log_none").exp() * np.exp(-miss)).alias("ef0"))
    kbest = (a.filter(pl.col("ef") == pl.col("ef_best")).group_by("s").agg(pl.col("k").min().alias("kb")))
    a = a.join(kbest, on="s").filter((pl.col("k") <= pl.col("kb")) & (pl.col("ef_best") > pl.col("ef0")))
    return a.select("q", "s", "p")


def decode(pp: pl.DataFrame, params: dict) -> pl.DataFrame:
    if params.get("mode") == "expected_f":
        return expected_f_prefix(pp, params["tau"], params["miss"])
    return assign(pp, params["tau"])


## 10. Training & validation

In [ ]:
"""Stage 4: fit the LightGBM pair matcher and tune the decoder on held-out S1s.

Validation protocol: a random VAL_S1_FRAC of train S1 is held out.  Every
S2/S3 query that has any held-out S1 among its candidates goes to validation
(with its full candidate list, so competition from other S1s is preserved);
the model is trained on a random sample of the remaining queries.
"""
import json
import sys
import time

import lightgbm as lgb
import numpy as np
import polars as pl
import polars.selectors as cs


PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=255, min_data_in_leaf=200,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              max_bin=255, num_threads=config.N_THREADS, seed=config.SEED, verbose=-1)


def gt_pairs(norm: pl.DataFrame) -> pl.DataFrame:
    ids = norm.select("entity_id", "idx")
    return (load_ground_truth()
            .join(ids.rename({"entity_id": "s1_id", "idx": "s"}), on="s1_id")
            .join(ids.rename({"entity_id": "match_id", "idx": "q"}), on="match_id")
            .select("s", "q").with_columns(pl.lit(1, pl.Int8).alias("label")))


def with_country(cand: pl.DataFrame, T: pl.DataFrame) -> pl.DataFrame:
    return cand.with_columns(pl.Series("country", T["country"].to_numpy()[cand["q"].to_numpy()]))


def make_sets(norm, cand):
    T = table(norm)
    gt = gt_pairs(norm)
    s1 = norm.filter(pl.col("src") == 1).select("idx", "country")
    val_s1 = s1.filter((pl.col("idx").hash(3) % 10_000) < config.VAL_S1_FRAC * 10_000)
    cand = with_country(cand, T)
    tr, va = [], []
    for c in cand["country"].unique().sort().to_list():
        t = time.time()
        cc = context_cos(cand.filter(pl.col("country") == c))
        cc = cc.with_columns(pl.col("s").is_in(val_s1["idx"].implode()).any().over("q").alias("isval"))
        samp = (pl.col("q").hash(11) % 10_000) < config.TRAIN_QUERY_FRAC * 10_000
        for dst, sub in ((tr, cc.filter(~pl.col("isval") & samp)), (va, cc.filter(pl.col("isval")))):
            f = build(sub, T, verbose=False)
            f = f.join(gt, on=["s", "q"], how="left").with_columns(pl.col("label").fill_null(0))
            dst.append(f.with_columns(cs.float().cast(pl.Float32)))
        print(f"[train] {c}: features done in {time.time() - t:.0f}s", flush=True)
        del cc
    return pl.concat(tr), pl.concat(va), gt, val_s1


def fit(tr: pl.DataFrame, va: pl.DataFrame, cols, rounds=3000):
    dtr = lgb.Dataset(tr.select(cols).to_numpy(), tr["label"].to_numpy(), feature_name=cols, free_raw_data=True)
    dva = lgb.Dataset(va.select(cols).to_numpy(), va["label"].to_numpy(), reference=dtr)
    return lgb.train(PARAMS, dtr, rounds, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])


def tune(va: pl.DataFrame, p: np.ndarray, gt: pl.DataFrame, val_s1: pl.DataFrame):
    pp = va.select("q", "s").with_columns(pl.Series("p", p))
    truth = gt.join(val_s1.select(pl.col("idx").alias("s")), on="s", how="semi")
    grid = [{"mode": "threshold", "tau": t} for t in np.arange(0.2, 0.95, 0.05)]
    grid += [{"mode": "expected_f", "tau": t, "miss": m} for t in (0.05, 0.1, 0.2, 0.3) for m in (0.0, 0.1, 0.25, 0.5, 1.0)]
    best, best_f = None, -1
    for g in grid:
        f = macro_f05(decode(pp, g), truth, val_s1["idx"])["macro_f05"]
        if f > best_f:
            best, best_f = g, f
    best = {k: float(v) if not isinstance(v, str) else v for k, v in best.items()}
    rep = {"overall": macro_f05(decode(pp, best), truth, val_s1["idx"])}
    for c in val_s1["country"].unique().to_list():
        ids = val_s1.filter(pl.col("country") == c)["idx"]
        rep[c] = macro_f05(decode(pp, best), truth, ids)
    return best, rep


def main(loco=False):
    norm = pl.read_parquet(config.WORK_DIR / "train_norm.parquet")
    cand = pl.read_parquet(config.WORK_DIR / "cand_train.parquet")
    feat_path = config.WORK_DIR / "train_feats.parquet", config.WORK_DIR / "val_feats.parquet"
    if all(p.exists() for p in feat_path):
        tr, va = (pl.read_parquet(p) for p in feat_path)
        gt = gt_pairs(norm)
        val_s1 = norm.filter((pl.col("src") == 1) & ((pl.col("idx").hash(3) % 10_000) < config.VAL_S1_FRAC * 10_000)).select("idx", "country")
    else:
        tr, va, gt, val_s1 = make_sets(norm, cand)
        tr.write_parquet(feat_path[0]); va.write_parquet(feat_path[1])
    del cand, norm
    cols = feature_cols(tr)
    print(f"[train] train pairs {tr.height:,} (pos {tr['label'].mean():.3f})  val pairs {va.height:,}", flush=True)

    # blocking recall ceiling on the validation S1s
    truth = gt.join(val_s1.select(pl.col("idx").alias("s")), on="s", how="semi")
    rec = truth.join(va.select("s", "q"), on=["s", "q"], how="semi").height / max(truth.height, 1)
    print(f"[train] validation blocking pair-recall {rec:.4f}", flush=True)

    if loco:  # leave-one-country-out: proxy for the unseen country (France)
        for held in va["country"].unique().to_list():
            b = fit(tr.filter(pl.col("country") != held), va.filter(pl.col("country") != held), cols, rounds=600)
            v = va.filter(pl.col("country") == held)
            best, rep = tune(v, b.predict(v.select(cols).to_numpy()), gt,
                             val_s1.filter(pl.col("country") == held))
            print(f"[loco] trained without {held}: {rep['overall']}  params {best}", flush=True)

    booster = fit(tr, va, cols)
    p = booster.predict(va.select(cols).to_numpy())
    best, rep = tune(va, p, gt, val_s1)
    for k, v in rep.items():
        print(f"[val] {k}: {v}", flush=True)
    print(f"[val] decode params {best}", flush=True)
    booster.save_model(str(config.WORK_DIR / "model.txt"))
    imp = sorted(zip(cols, booster.feature_importance("gain")), key=lambda x: -x[1])
    json.dump({"decode": best, "features": cols, "val": rep, "blocking_recall": rec,
               "importance": [(c, float(g)) for c, g in imp]},
              open(config.WORK_DIR / "train_report.json", "w"), indent=1, default=str)


In [ ]:
main(loco=False)  # set loco=True for the leave-one-country-out check (France proxy)

## 11. Test prediction & output files

In [ ]:
"""Stage 6: score test candidates, decode and write output/*.tsv."""
import json
import time

import lightgbm as lgb
import numpy as np
import polars as pl


Q_CHUNK = 400_000  # queries per feature chunk


def score(split="test", booster=None, cols=None) -> pl.DataFrame:
    norm = pl.read_parquet(config.WORK_DIR / f"{split}_norm.parquet")
    cand = pl.read_parquet(config.WORK_DIR / f"cand_{split}.parquet")
    rep = json.load(open(config.WORK_DIR / "train_report.json"))
    booster = booster or lgb.Booster(model_file=str(config.WORK_DIR / "model.txt"))
    cols = cols or rep["features"]
    T = table(norm)
    cand = with_country(cand, T)
    out = []
    for c in cand["country"].unique().sort().to_list():
        t = time.time()
        cc = context_cos(cand.filter(pl.col("country") == c)).sort("q")
        qv = cc["q"].to_numpy()
        cuts = np.searchsorted(qv, np.unique(qv)[::Q_CHUNK].tolist() + [qv[-1] + 1])
        for a, b in zip(cuts[:-1], cuts[1:]):
            f = build(cc.slice(a, b - a), T, verbose=False)
            p = booster.predict(f.select(cols).to_numpy().astype(np.float32))
            out.append(f.select("q", "s").with_columns(pl.Series("p", p.astype(np.float32))))
        print(f"[predict] {c}: {cc.height:,} pairs scored in {time.time() - t:.0f}s", flush=True)
        del cc
    pp = pl.concat(out)
    pp.write_parquet(config.WORK_DIR / f"scores_{split}.parquet")
    return pp


def write_outputs(pp: pl.DataFrame, split="test"):
    norm = pl.read_parquet(config.WORK_DIR / f"{split}_norm.parquet", columns=["idx", "entity_id", "src", "country"])
    params = json.load(open(config.WORK_DIR / "train_report.json"))["decode"]
    matches = decode(pp, params)
    ids = norm.select(pl.col("idx").alias("q"), pl.col("entity_id").alias("q_id"))
    s1 = norm.filter(pl.col("src") == 1).sort("idx").select("idx", "entity_id", "country")
    config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write_lists(config.OUTPUT_DIR / "matching_results.tsv", s1,
                matches.join(ids, on="q").select("s", "q_id"), "matched_entity_ids")
    write_lists(config.OUTPUT_DIR / "candidate_pairs.tsv", s1,
                pp.join(ids, on="q").select("s", "q_id"), "candidate_entity_ids")
    # summary per country (sanity check for the unlabeled France part)
    per = (s1.join(matches.group_by("s").len("n"), left_on="idx", right_on="s", how="left").fill_null(0)
             .group_by("country").agg(pl.len().alias("s1"), pl.col("n").mean().alias("avg_matches"),
                                      (pl.col("n") == 0).mean().alias("empty_rate")))
    print(per, flush=True)


In [ ]:
pp = score('test')
write_outputs(pp, 'test')

## 12. Validate submission format

In [ ]:
# format checks (same rules as utils/validate_submission.py; runs that script too if present)
import glob, subprocess
m = pl.read_csv(config.OUTPUT_DIR / "matching_results.tsv", separator="\t", quote_char=None, infer_schema=False, missing_utf8_is_empty_string=True)
c = pl.read_csv(config.OUTPUT_DIR / "candidate_pairs.tsv", separator="\t", quote_char=None, infer_schema=False, missing_utf8_is_empty_string=True)
s1 = read_tsv(config.DATA_DIR / "test" / "test_source1.tsv")["entity_id"]
assert config.SAMPLE_FRAC or m.height == s1.len() and m["source1_entity_id"].n_unique() == m.height and set(m["source1_entity_id"]) == set(s1)
ml = m.select("source1_entity_id", pl.col("matched_entity_ids").fill_null("").str.split(",")).explode("matched_entity_ids").filter(pl.col("matched_entity_ids") != "")
cl = c.select("source1_entity_id", pl.col("candidate_entity_ids").fill_null("").str.split(",")).explode("candidate_entity_ids").filter(pl.col("candidate_entity_ids") != "")
assert ml.filter(~pl.col("matched_entity_ids").str.contains("^S[23]-")).height == 0
assert ml.height == ml.unique().height and cl.height == cl.unique().height
assert ml.join(cl.rename({"candidate_entity_ids": "matched_entity_ids"}), on=["source1_entity_id", "matched_entity_ids"], how="anti").height == 0
print(f"format OK: {m.height:,} S1 rows, {ml.height:,} matches, {cl.height:,} candidates")
v = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
if v:
    print(subprocess.run(["python3", v[0], "--matching", str(config.OUTPUT_DIR / "matching_results.tsv"), "--candidate", str(config.OUTPUT_DIR / "candidate_pairs.tsv"), "--test-dir", str(config.DATA_DIR / "test")], capture_output=True, text=True).stdout)